# MaNGA drpall HEALPix collision check (mirrors MMU build_parent_sample.py)

Reproduce MMU's parent sample exactly:
- `drpall-v3_1_1.fits` MANGA HDU
- inner-join with dapall HYB10-MILESHC-MASTARSSP rows on plateifu / PLATEIFU
- keep `DAPDONE == True`

Then run the (objra, objdec) collision sweep on that filtered sample.

In [1]:
from collections import Counter

import healpy as hp
import numpy as np
from astropy.table import Table, join

DRPALL = '/mnt/home/thehir/tmp-catalog-downloads/drpall-v3_1_1.fits'
DAPALL = '/mnt/home/thehir/tmp-catalog-downloads/dapall-v3_1_1-3.1.0.fits'
DAPTYPE = 'HYB10-MILESHC-MASTARSSP'

## Load drpall MANGA + dapall, then mirror MMU's filter

MMU reads the named HDU `HYB10-MILESHC-MASTARSSP` directly. List dapall's HDUs first to confirm it's there.

In [2]:
import astropy.io.fits as fits

drp = Table.read(DRPALL, hdu='MANGA')
print(f'drpall MANGA rows: {len(drp)}')

with fits.open(DAPALL) as h:
    print(f'dapall HDUs: {[hdu.name for hdu in h]}')

dap = Table.read(DAPALL, hdu=DAPTYPE)
print(f'dapall {DAPTYPE} rows: {len(dap)}')

drpall MANGA rows: 11273
dapall HDUs: ['PRIMARY', 'SPX-MILESHC-MASTARSSP', 'VOR10-MILESHC-MASTARSSP', 'HYB10-MILESHC-MASTARSSP', 'HYB10-MILESHC-MASTARHC2']
dapall HYB10-MILESHC-MASTARSSP rows: 10782


In [3]:
# Mirror MMU exactly: inner-join on plateifu, then DAPDONE filter.
cat = join(drp, dap, keys_left='plateifu', keys_right='PLATEIFU', join_type='inner')
print(f'after inner join: {len(cat)}')

cat = cat[cat['DAPDONE']]
print(f'after DAPDONE filter: {len(cat)}')

after inner join: 10782
after DAPDONE filter: 10735


## Collision sweep on the MMU parent sample

In [4]:
RA_COL = 'objra'
DEC_COL = 'objdec'
ra = np.asarray(cat[RA_COL], dtype=np.float64)
dec = np.asarray(cat[DEC_COL], dtype=np.float64)
finite = np.isfinite(ra) & np.isfinite(dec)
print(f'finite coords: {finite.sum()}/{len(ra)}')
ra, dec = ra[finite], dec[finite]

mangaids = [x.decode() if isinstance(x, bytes) else str(x) for x in cat['mangaid'][finite]]
plateifus = [x.decode() if isinstance(x, bytes) else str(x) for x in cat['plateifu'][finite]]
print(f'unique mangaid: {len(set(mangaids))}')
print(f'unique plateifu: {len(set(plateifus))}')

coord_counts = Counter(zip(ra.tolist(), dec.tolist()))
dup_coords = [(r, d, n) for (r, d), n in coord_counts.items() if n > 1]
print(f'bit-identical (objra, objdec) groups: {len(dup_coords)}')
print(f'total rows in identical groups: {sum(n for _, _, n in dup_coords)}')
print(f'max rows at one coordinate: {max((n for _, _, n in dup_coords), default=0)}')

finite coords: 10735/10735
unique mangaid: 10555
unique plateifu: 10735
bit-identical (objra, objdec) groups: 145
total rows in identical groups: 317
max rows at one coordinate: 5


In [5]:
for order in range(5, 30):
    nside = 2 ** order
    pix = hp.ang2pix(nside, ra, dec, lonlat=True, nest=True)
    _, counts = np.unique(pix, return_counts=True)
    print(f'order={order:2d} max_per_pixel={counts.max():5d} n_collisions={(counts>1).sum()}')

order= 5 max_per_pixel=  345 n_collisions=1038
order= 6 max_per_pixel=  229 n_collisions=2274
order= 7 max_per_pixel=  226 n_collisions=2274
order= 8 max_per_pixel=  117 n_collisions=1410
order= 9 max_per_pixel=  117 n_collisions=726
order=10 max_per_pixel=   62 n_collisions=375
order=11 max_per_pixel=   22 n_collisions=235
order=12 max_per_pixel=    6 n_collisions=256
order=13 max_per_pixel=    5 n_collisions=213
order=14 max_per_pixel=    5 n_collisions=158
order=15 max_per_pixel=    5 n_collisions=154
order=16 max_per_pixel=    5 n_collisions=152
order=17 max_per_pixel=    5 n_collisions=151
order=18 max_per_pixel=    5 n_collisions=149
order=19 max_per_pixel=    5 n_collisions=148
order=20 max_per_pixel=    5 n_collisions=147
order=21 max_per_pixel=    5 n_collisions=146
order=22 max_per_pixel=    5 n_collisions=144
order=23 max_per_pixel=    5 n_collisions=144
order=24 max_per_pixel=    5 n_collisions=145
order=25 max_per_pixel=    5 n_collisions=145
order=26 max_per_pixel=    5 n

## Cross-check against the HDF5 staging histogram

In [6]:
HIST = '/mnt/ceph/users/polymathic/manga-staging-tmp/mmu_manga/intermediate/mmu_manga/intermediate/row_count_mapping_histogram.npz'
with open(HIST, 'rb') as f:
    full = np.frombuffer(f.read(), dtype=np.int64)
print(f'hdf5 rows (histogram): {full.sum()}')
print(f'mmu parent sample rows: {len(cat)}')
print(f'diff: parent - hdf5 = {len(cat) - full.sum()}')

hdf5 rows (histogram): 10735
mmu parent sample rows: 10735
diff: parent - hdf5 = 0
